In [12]:
import os
import cv2
import zipfile
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

In [17]:
ZIP_FILE_NAME = "anti_spoofing.zip"

with zipfile.ZipFile(ZIP_FILE_NAME, 'r') as zip_ref:
    zip_ref.extractall("dataset")

print("Dataset extracted successfully.")

Dataset extracted successfully.


In [58]:
IMG_SIZE = 64
BATCH_SIZE = 32
EPOCHS = 30

DATASET_PATH = "/content/dataset"

VIDEO_EXT = (".mp4", ".avi", ".mov", ".mkv")

REAL_FOLDERS = ["live_selfie", "live_video"]
FAKE_FOLDERS = ["replay", "printouts", "cut-out_printouts"]

In [44]:
def extract_frames(video_path, max_frames=8):
    frames = []

    cap = cv2.VideoCapture(video_path)
    count = 0

    while cap.isOpened() and count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        frame = frame.astype("float32") / 255.0

        frames.append(frame)
        count += 1

    cap.release()
    return frames

In [59]:
def load_data():
    images = []
    labels = []

    print("Loading dataset...")

    for folder in os.listdir(DATASET_PATH):
        folder_path = os.path.join(DATASET_PATH, folder)

        if not os.path.isdir(folder_path):
            continue

        if folder in REAL_FOLDERS:
            label = 1
        elif folder in FAKE_FOLDERS:
            label = 0
        else:
            continue

        print(f"Processing: {folder}")

        for file in os.listdir(folder_path):
            file_path = os.path.join(folder_path, file)

            try:

                if file.lower().endswith((".jpg", ".jpeg", ".png")):
                    img = cv2.imread(file_path)
                    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                    img = img.astype("float32") / 255.0

                    images.append(img)
                    labels.append(label)

                elif file.lower().endswith(VIDEO_EXT):
                    frames = extract_frames(file_path)

                    for frame in frames:
                        images.append(frame)
                        labels.append(label)

            except Exception as e:
                print("Skipping:", file_path, e)

    return np.array(images), np.array(labels)

In [60]:
X, y = load_data()

print("Total samples:", len(X))

Loading dataset...
Processing: live_video
Processing: printouts
Processing: live_selfie
Processing: replay
Total samples: 225


In [61]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [53]:
model = Sequential([

    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(0.2),

    Conv2D(32, (3,3), activation="relu", input_shape=(64, 64, 3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(128, activation="relu"),
    Dropout(0.5),

    Dense(1, activation="sigmoid")
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [64]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip (RandomFlip)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 64, 64, 3)      │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast                 │ (None, 64, 64, 3)      │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_brightness               │ (None, 64, 64, 3)      │             0 │
│ (RandomBrightness)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 683,329 (2.61 MB)

 Trainable params: 683,329 (2.61 MB)

 Non-trainable params: 0 (0.00 B)

In [62]:
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

Epoch 1/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.6444 - loss: 0.6551 - val_accuracy: 0.6444 - val_loss: 0.6008
Epoch 2/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.6444 - loss: 0.6499 - val_accuracy: 0.6444 - val_loss: 0.5988
Epoch 3/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.6500 - loss: 0.6581 - val_accuracy: 0.6444 - val_loss: 0.5956
Epoch 4/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.6389 - loss: 0.6618 - val_accuracy: 0.6444 - val_loss: 0.5891
Epoch 5/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.6444 - loss: 0.6539 - val_accuracy: 0.6444 - val_loss: 0.5889
Epoch 6/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.6500 - loss: 0.6687 - val_accuracy: 0.6444 - val_loss: 0.5894
Epoch 7/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.6444 - loss: 0.6579 - val_accuracy: 0.6444 - val_loss: 0.5887
Epoch 8/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.6278 - loss: 0.6760 - val_accuracy: 0.6444 - val_loss: 0.5887


In [63]:
model.save("anti_spoof_model.keras")

print("Done.")

Done.
